# 01 — Agent Basics

**No API key needed.** Pure Python + LangGraph.

---

## What is an Agent?

An **agent** is a program that:
1. **Perceives** its environment (reads state)
2. **Decides** what to do next (uses logic or an LLM)
3. **Acts** using tools
4. **Loops** until the goal is reached

```
 ┌─────────────────────────────────────────┐
 │              Agent Loop                 │
 │                                         │
 │  Perceive State                         │
 │       │                                 │
 │       ▼                                 │
 │  Decide (rule / LLM)                    │
 │       │                                 │
 │       ▼                                 │
 │  Act  (call a tool)                     │
 │       │                                 │
 │       ▼                                 │
 │  Update State ──────────────────────────┤
 │                           (or → END)    │
 └─────────────────────────────────────────┘
```

In this notebook we build a **Task Agent** that:
- Holds a to-do list in its state
- Picks the next task
- Routes it to the right tool (calculator or weather)
- Marks it done and loops until the list is empty

**Concepts covered:**
| Concept | What it means |
|---------|---------------|
| `TypedDict` State | The shared data object that flows through every node |
| Node | A Python function that reads state and returns updates |
| Conditional edge | Routes to different nodes based on the current state |
| Agent loop | A graph edge that points back to an earlier node |

## Step 1 — Install dependencies

Run this once, then restart the kernel.

In [ ]:
# %pip install langgraph langchain-core --quiet

## Step 2 — Define the Agent State

State is the **single source of truth** for the agent.  
Every node reads from it and returns a *partial update*.

In [ ]:
from typing import TypedDict, List, Optional


class Task(TypedDict):
    id: int
    type: str   # "calculate" or "weather"
    input: str


class AgentState(TypedDict):
    tasks: List[Task]          # pending tasks
    current_task: Optional[Task]  # task being processed right now
    results: List[str]         # completed results
    status: str                # "running" | "done"


print("State schema defined.")
print("Fields:", list(AgentState.__annotations__.keys()))

## Step 3 — Define the Tools

Tools are plain Python functions. The agent calls them to interact with the world.

In [ ]:
def calculator_tool(expression: str) -> str:
    """Safely evaluate a math expression and return the result."""
    try:
        # eval is safe here because we only allow digits and operators
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return f"Error: invalid characters in '{expression}'"
        result = eval(expression)  # noqa: S307
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"


def weather_tool(city: str) -> str:
    """Return a mock weather report for any city."""
    mock_db = {
        "london": "Cloudy, 14°C",
        "tokyo": "Sunny, 28°C",
        "new york": "Rainy, 18°C",
        "paris": "Partly cloudy, 20°C",
        "sydney": "Clear, 22°C",
    }
    report = mock_db.get(city.lower(), "Weather data unavailable")
    return f"{city}: {report}"


# Quick test
print(calculator_tool("(15 + 7) * 3"))
print(weather_tool("Tokyo"))

## Step 4 — Define the Agent Nodes

Each **node** is a Python function that:
- Receives the full state as input
- Returns a **dict** with only the fields it wants to update

In [ ]:
def pick_next_task(state: AgentState) -> dict:
    """Pick the first pending task and set it as current."""
    if not state["tasks"]:
        # No tasks left — signal completion
        return {"current_task": None, "status": "done"}

    # Pop the first task from the queue
    remaining = list(state["tasks"])
    next_task = remaining.pop(0)

    print(f"  [pick_next_task] picked → Task #{next_task['id']}: {next_task['type']}")
    return {"tasks": remaining, "current_task": next_task, "status": "running"}


def run_calculator(state: AgentState) -> dict:
    """Execute the calculator tool on the current task."""
    task = state["current_task"]
    result = calculator_tool(task["input"])
    print(f"  [run_calculator] result → {result}")
    return {"results": state["results"] + [result]}


def run_weather(state: AgentState) -> dict:
    """Execute the weather tool on the current task."""
    task = state["current_task"]
    result = weather_tool(task["input"])
    print(f"  [run_weather] result → {result}")
    return {"results": state["results"] + [result]}


print("Nodes defined: pick_next_task, run_calculator, run_weather")

## Step 5 — Define the Routing Logic

**Conditional edges** are functions that return a node name.  
LangGraph uses the return value to decide where to go next.

In [ ]:
def route_task(state: AgentState) -> str:
    """
    After pick_next_task:
    - If done → go to END
    - If task type is 'calculate' → go to run_calculator
    - If task type is 'weather'   → go to run_weather
    """
    if state["status"] == "done":
        return "__end__"

    task_type = state["current_task"]["type"]
    if task_type == "calculate":
        return "run_calculator"
    elif task_type == "weather":
        return "run_weather"
    else:
        # Unknown task type — skip it and loop back
        return "pick_next_task"


print("Routing function defined.")

## Step 6 — Build the Graph

Now we wire the nodes and edges into a `StateGraph`.

```
START
  │
  ▼
pick_next_task
  │
  ├─── (done) ──────────────────── END
  │
  ├─── (calculate) ─▶ run_calculator ─▶ pick_next_task (loop)
  │
  └─── (weather)   ─▶ run_weather   ─▶ pick_next_task (loop)
```

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Create the graph with our state schema
graph = StateGraph(AgentState)

# 2. Register nodes
graph.add_node("pick_next_task", pick_next_task)
graph.add_node("run_calculator", run_calculator)
graph.add_node("run_weather", run_weather)

# 3. Entry point
graph.add_edge(START, "pick_next_task")

# 4. Conditional edge from pick_next_task — routes based on task type
graph.add_conditional_edges(
    "pick_next_task",
    route_task,
    {
        "run_calculator": "run_calculator",
        "run_weather": "run_weather",
        "__end__": END,
    },
)

# 5. After each tool runs, loop back to pick the next task
graph.add_edge("run_calculator", "pick_next_task")
graph.add_edge("run_weather", "pick_next_task")

# 6. Compile
agent = graph.compile()

print("Graph compiled successfully.")

## Step 7 — Run the Agent

Create an initial state with a mix of tasks and watch the agent loop through them.

In [ ]:
initial_state: AgentState = {
    "tasks": [
        {"id": 1, "type": "calculate", "input": "(100 + 50) / 5"},
        {"id": 2, "type": "weather",   "input": "London"},
        {"id": 3, "type": "calculate", "input": "7 * 8 + 3"},
        {"id": 4, "type": "weather",   "input": "Tokyo"},
        {"id": 5, "type": "calculate", "input": "2.5 * 4"},
    ],
    "current_task": None,
    "results": [],
    "status": "running",
}

print("Running agent...\n")
final_state = agent.invoke(initial_state)

print("\n" + "=" * 45)
print("FINAL RESULTS")
print("=" * 45)
for i, result in enumerate(final_state["results"], 1):
    print(f"  {i}. {result}")
print(f"\nStatus: {final_state['status']}")
print(f"Tasks remaining: {len(final_state['tasks'])}")

## Step 8 — Visualise the Graph

LangGraph can draw the graph structure for you.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png requires extra dependencies
    print(agent.get_graph().draw_mermaid())

## Step 9 — Try It Yourself

Add your own tasks to the list and re-run the agent.

In [ ]:
my_tasks: AgentState = {
    "tasks": [
        # Add your own tasks here
        {"id": 1, "type": "calculate", "input": "99 * 99"},
        {"id": 2, "type": "weather",   "input": "Paris"},
        {"id": 3, "type": "weather",   "input": "Sydney"},
    ],
    "current_task": None,
    "results": [],
    "status": "running",
}

result = agent.invoke(my_tasks)
print("\nResults:")
for r in result["results"]:
    print(f"  • {r}")

## Summary

| What you built | What it taught |
|---------------|----------------|
| `AgentState` TypedDict | Every agent needs a shared state object |
| `pick_next_task` node | Agents perceive state to decide what to do next |
| `route_task` conditional edge | Routing logic separates decisions from actions |
| Loop back to `pick_next_task` | The agent loop: keep acting until the goal is met |

---

**Next →** [02 — Tool Calling Agent](../02-tool-calling-agent/tool_calling_agent.ipynb)  
Add a real LLM (Groq) that decides *which tool to call* based on a natural language question.